# Pipeline Completo - Clustering de Olhos Barrett II

## 8 Fases: Loading → EDA → Preprocessing → Find K → K-Means → DBSCAN → Hierarchical → Validation

In [ ]:
# ============================================================
# FASE 0: CARREGAMENTO DE DADOS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/home/antonio/Documentos/Machine Learning/atividade_3/barrettII_eyes_clustering.csv'

df = pd.read_csv(DATA_PATH, decimal=',')
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head()

In [ ]:
# Limpeza
numeric_cols = ['AL', 'ACD', 'WTW', 'K1', 'K2']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=numeric_cols)
print(f'Shape apos limpeza: {df.shape}')

df_clean = df.copy()
df_biometric = df_clean[numeric_cols].copy()

In [ ]:
# ============================================================
# FASE 1: ANALISE EXPLORATORIA (EDA)
# ============================================================

# Estatisticas descritivas
stats = df_biometric.describe().T
stats['median'] = df_biometric.median()
stats['variance'] = df_biometric.var()
stats['skewness'] = df_biometric.skew()
stats['kurtosis'] = df_biometric.kurtosis()
print('Estatisticas Descritivas:')
print(stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(3))

In [ ]:
# Histogramas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    ax.hist(df_biometric[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(df_biometric[col].mean(), color='red', linestyle='--', label=f'Media: {df_biometric[col].mean():.2f}')
    ax.axvline(df_biometric[col].median(), color='green', linestyle='--', label=f'Mediana: {df_biometric[col].median():.2f}')
    ax.set_title(f'Histograma {col}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

axes[5].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
fig, ax = plt.subplots(figsize=(12, 6))
df_biometric.boxplot(ax=ax, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
ax.set_title('Boxplot - Variaveis Biometricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de Correlacao
corr_matrix = df_biometric.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlacao', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FASE 2: PREPROCESSAMENTO
# ============================================================

from sklearn.preprocessing import StandardScaler, MinMaxScaler

X = df_biometric.values

# StandardScaler (Z-Score)
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
df_std = pd.DataFrame(X_std, columns=numeric_cols)
print(f'Dados Originais: shape={X.shape}')
print(f'X_standard: mean~0, std~1')
print(f'X_minmax: range=[0,1]')

In [ ]:
# ============================================================
# FASE 3: ENCONTRAR K OTIMO
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 16)
inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_std)
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(X_std, kmeans.labels_)
    silhouettes.append(sil)
    print(f'K={k}: inertia={kmeans.inertia_:.2f}, silhouette={sil:.4f}')

In [ ]:
# Elbow + Silhouette plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Metodo do Cotovelo', fontweight='bold')
axes[0].grid(True, alpha=0.3)

best_k = list(k_range)[np.argmax(silhouettes)]
axes[1].plot(k_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='red', linestyle='--', label=f'Melhor K={best_k}')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Analise de Silhueta', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nMelhor K pelo Silhouette: {best_k}')

In [ ]:
# ============================================================
# FASE 4: K-MEANS CLUSTERING
# ============================================================

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

optimal_k = 2
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_std)

print(f'K-Means com K={optimal_k}')
print(f'Cluster sizes: {np.bincount(kmeans_labels)}')

In [ ]:
# PCA + t-SNE visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_std)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.6, s=30)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroides')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title(f'K-Means (K={optimal_k}) - PCA', fontweight='bold')
axes[0].legend()
plt.colorbar(scatter, ax=axes[0], label='Cluster')

scatter = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title(f'K-Means (K={optimal_k}) - t-SNE', fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

print('Labels K-Means salvos!')

In [ ]:
# ============================================================
# FASE 5: DBSCAN CLUSTERING
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# K-distance graph
k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_std)
distances, indices = nn.kneighbors(X_std)
distances = np.sort(distances[:, k-1])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(distances, 'b-', linewidth=1)
ax.set_xlabel('Pontos (ordenados)')
ax.set_ylabel(f'Distancia ao {k}-vizinho mais proximo')
ax.set_title('Grafico K-Distancia (para encontrar eps)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Testar diferentes eps
print('--- DBSCAN com diferentes eps ---')
for eps in [0.5, 1.0, 1.5, 2.0, 2.5]:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_std)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    print(f'eps={eps}: {n_clusters} clusters, {n_noise} noise points')

In [ ]:
# DBSCAN final
optimal_eps = 1.5
dbscan = DBSCAN(eps=optimal_eps, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_std)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f'\nDBSCAN final (eps={optimal_eps}):')
print(f'  Clusters: {n_clusters}')
print(f'  Noise: {n_noise} ({n_noise/len(dbscan_labels)*100:.1f}%)')

In [ ]:
# ============================================================
# FASE 6: HIERARCHICAL CLUSTERING
# ============================================================

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Linkage matrix
linkage_matrix = linkage(X_std, method='ward')

# Dendrograma
fig, ax = plt.subplots(figsize=(16, 8))
dendrogram(linkage_matrix, ax=ax, truncate_mode='lastp', p=50,
           leaf_rotation=90, leaf_font_size=10, show_contracted=True)
ax.set_title('Dendrograma - Ward Linkage', fontsize=14, fontweight='bold')
ax.set_xlabel('Amostra ou Cluster ID')
ax.set_ylabel('Distancia (Ward)')
ax.axhline(y=50, color='r', linestyle='--', label='Corte para 2 clusters')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Hierarchical com 2 clusters
hc = AgglomerativeClustering(n_clusters=2, linkage='ward')
hc_labels = hc.fit_predict(X_std)

print(f'Hierarchical (2 clusters):')

In [ ]:
print(f'  Cluster sizes: {np.bincount(hc_labels)}')

In [ ]:
# ============================================================
# FASE 7: VALIDACAO E COMPARACAO
# ============================================================

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.stats import chi2_contingency

# Funcao para calcular metricas
def compute_metrics(X, labels, method_name):
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    mask = labels != -1
    labels_filter = labels[mask]
    X_filter = X[mask]
    unique_filtered = np.unique(labels_filter)
    n_clusters_filtered = len(unique_filtered)
    if n_clusters_filtered < 2:
        return {'Method': method_name, 'N_clusters': n_clusters, 'Silhouette': np.nan,
                'Calinski-Harabasz': np.nan, 'Davies-Bouldin': np.nan}
    sil = silhouette_score(X_filter, labels_filter)
    ch = calinski_harabasz_score(X_filter, labels_filter)
    db = davies_bouldin_score(X_filter, labels_filter)
    return {'Method': method_name, 'N_clusters': n_clusters,
            'Silhouette': sil, 'CH': ch, 'DB': db}

In [ ]:
# Calcular metricas para todos metodos
metrics = []
metrics.append(compute_metrics(X_std, kmeans_labels, 'K-Means (K=2)'))
metrics.append(compute_metrics(X_std, dbscan_labels, 'DBSCAN'))
metrics.append(compute_metrics(X_std, hc_labels, 'Hierarchical (Ward)'))

results_df = pd.DataFrame(metrics)
results_df['Silhouette'] = results_df['Silhouette'].round(4)
results_df['CH'] = results_df['CH'].round(2)
results_df['DB'] = results_df['DB'].round(4)

print('=== VALIDACAO - METRICAS DE CLUSTERING ===')
print(results_df.to_string(index=False))

In [ ]:
# Criar profiles para radar
df_kmeans = df_std.copy()
df_kmeans['Cluster'] = kmeans_labels
profile_kmeans = df_kmeans.groupby('Cluster').mean()

print('=== K-MEANS CLUSTER PROFILES ===')
print(profile_kmeans.round(3))

In [ ]:
# Radar Chart
cluster_0 = profile_kmeans.loc[0].values
cluster_1 = profile_kmeans.loc[1].values

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
n_vars = len(numeric_cols)
angles = np.linspace(0, 2 * np.pi, n_vars, endpoint=False).tolist()
angles += angles[:1]

ax.plot(angles, cluster_0.tolist() + [cluster_0[0]], 'o-', linewidth=2, label='Cluster 0', color='blue')
ax.fill(angles, cluster_0.tolist() + [cluster_0[0]], alpha=0.25, color='blue')
ax.plot(angles, cluster_1.tolist() + [cluster_1[0]], 'o-', linewidth=2, label='Cluster 1', color='red')
ax.fill(angles, cluster_1.tolist() + [cluster_1[0]], alpha=0.25, color='red')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(numeric_cols, fontsize=12)
ax.set_title('K-Means: Cluster Profiles (Radar)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation com variavel Correto
df_clean['KMeans_Cluster'] = kmeans_labels
cross = pd.crosstab(df_clean['Correto'], df_clean['KMeans_Cluster'])

print('=== CROSS: Correto vs K-Means ===')
print(cross)

chi2, p, dof, expected = chi2_contingency(cross)
print(f'\nChi2={chi2:.2f}, p-value={p:.4f}')

In [ ]:
# ============================================================
# CONCLUSAO
# ============================================================

# Contagens dos clusters
cluster_counts = pd.Series(kmeans_labels).value_counts().sort_index()

print('='*60)
print('RESULTADO FINAL - K-MEANS COM K=2')
print('='*60)
print(f'\nCluster 0 (Perfil A - Olhos Compactos): {cluster_counts[0]} olhos')
print(f'Cluster 1 (Perfil B - Olhos Alongados): {cluster_counts[1]} olhos')
print(f'Total: {len(kmeans_labels)} olhos')

# Caracteristicas dos perfis
df_profiles = df_biometric.copy()
df_profiles['Cluster'] = kmeans_labels

print('\n--- Caracteristicas Medias (dados originais) ---')
profiles_orig = df_profiles.groupby('Cluster').mean()
print(profiles_orig.round(3))

In [ ]:
# Perfil A vs Perfil B
print('\n--- INTERPRETACAO DOS PERFIS ---')
print('\nPerfil A (Cluster 0) - Olhos Compactos:')
print('  - AL menor (olho mais curto)')
print('  - WTW menor (cornea menor)')
print('  - K1/K2 maiores (cornea mais curva)')
print('  - Implicacao: Atencao em calculos de LIO')

print('\nPerfil B (Cluster 1) - Olhos Alongados:')
print('  - AL maior (olho mais longo)')
print('  - WTW maior (cornea maior)')
print('  - K1/K2 menores (cornea mais plana)')
print('  - Implicacao: Protocolos de medicao ajustados')

print('\n' + '='*60)
print('METRICAS DE VALIDACAO')
print('='*60)
print(results_df.to_string(index=False))